# Qwen3-0.6B + LoRA 微调: 中文医学诊疗指南文本生成

> 完整的训练流程笔记本 — 可独立运行

本笔记本包含:
1. 环境配置与镜像设置
2. 数据加载与预处理
3. 模型加载与 LoRA 配置
4. 训练循环与评估
5. 推理生成

硬件需求: GPU ≥4GB 显存 (或 CPU ≥8GB 内存)

## 0. 环境配置

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import re
import json
import time
import random
import logging
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

from importlib.metadata import version
for pkg in ["torch", "transformers", "peft", "datasets"]:
    print(f"{pkg}: {version(pkg)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. 超参数配置

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_LENGTH = 512
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 5
LEARNING_RATE = 2e-4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
DATA_DIR = "./data"
OUTPUT_DIR = "./output"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 2. 数据准备

支持两种方式:
- **真实数据**: 指定 `raw_data/` 目录含 .md 文件
- **样本数据**: 自动生成医学模板文本用于测试

In [ ]:
def generate_sample_data(num_samples=100):
    templates = {
        "临床表现": [
            "临床表现：患者男性，{age}岁，因\"{s1}伴{s2}{dur}\"入院。"
            "患者{dur}前无明显诱因出现{s1}，呈{qual}，伴{s2}，"
            "无{neg}等不适。查体：体温37.2℃，脉搏80次/分，"
            "呼吸18次/分，血压120/80mmHg。{finding}。",
        ],
        "诊断依据": [
            "诊断依据：1. 临床表现：患者有{s1}和{s2}的典型表现。"
            "2. 实验室检查：血常规示白细胞计数{wbc}×10⁹/L，CRP{crp}mg/L。"
            "3. 影像学检查：{img_type}示{img_find}。"
            "4. 病理学检查：{path}提示{diag}。综合以上，{diag}诊断明确。",
        ],
        "治疗方案": [
            "治疗方案：1. {tx}治疗：{drug}，{dose}，{route}，{freq}。"
            "2. 对症支持治疗：根据患者{s1}情况，给予{support}。"
            "3. 治疗期间需密切监测{monitor}。",
        ],
        "预后判断": [
            "预后判断：该患者为{stage}期{diag}，病理分级{grade}。"
            "综合评估：{risk}风险组。五年生存率约为{rate}%，"
            "中位生存时间约{med}个月。建议{follow}。",
        ],
        "鉴别诊断": [
            "鉴别诊断：1. {dd1}：{diff1}。"
            "2. {dd2}：{diff2}。"
            "3. {dd3}：{diff3}。通过{method}可资鉴别。",
        ],
        "用药原则": [
            "用药原则：1. 严格掌握{drug_type}的适应证和禁忌证。"
            "2. 注意药物相互作用：{drug}与{drug2}联合使用时{interaction}。"
            "3. 个体化给药：根据患者{factor}调整剂量。"
            "4. 监测不良反应：定期复查{monitor_items}。",
        ],
    }
    fillers = {
        "age": ["45", "52", "63", "38", "71", "56"],
        "s1": ["发热", "咳嗽", "腹痛", "头痛", "胸痛", "恶心", "乏力"],
        "s2": ["呕吐", "咳痰", "腹泻", "心悸", "盗汗", "食欲减退", "失眠"],
        "dur": ["1周", "3天", "2个月", "半年", "1个月", "10天"],
        "qual": ["持续性钝痛", "间歇性绞痛", "进行性加重", "反复发作性"],
        "neg": ["发热", "呕吐", "呼吸困难"],
        "finding": ["腹部有压痛，无反跳痛", "可触及肿块", "听诊呼吸音粗"],
        "wbc": ["8.5", "12.3", "6.7", "15.8"],
        "crp": ["25", "48", "12", "67"],
        "img_type": ["胸部CT", "腹部超声", "MRI"],
        "img_find": ["见占位性病变", "示多发结节影", "可见淋巴结增大"],
        "path": ["穿刺活检", "手术切除标本"],
        "diag": ["原发性支气管肺癌", "胃腺癌", "结直肠癌", "肝细胞癌"],
        "tx": ["手术", "化学", "放射", "靶向"],
        "drug": ["顺铂", "紫杉醇", "吉西他滨", "奥沙利铂"],
        "dose": ["75mg/m²", "175mg/m²", "1000mg/m²"],
        "route": ["静脉滴注", "口服", "静脉注射"],
        "freq": ["每3周一次", "每日一次", "每周一次"],
        "support": ["止吐、补液", "营养支持", "镇痛治疗"],
        "monitor": ["血常规、肝肾功能", "心电图、心肌酶谱", "肿瘤标志物"],
        "stage": ["I", "II", "III", "IV"],
        "grade": ["中分化", "低分化", "高分化"],
        "risk": ["低", "中", "高"],
        "rate": ["30", "45", "60", "75"],
        "med": ["12", "18", "24", "36"],
        "follow": ["每3个月复查一次", "每6个月全面评估", "终身随访"],
        "dd1": ["肺炎", "肺结核", "间质性肺病"],
        "dd2": ["食管癌", "淋巴瘤", "纵隔肿瘤"],
        "dd3": ["肺转移瘤", "肺栓塞", "结节病"],
        "diff1": ["临床表现相似，但影像学有特征性改变", "可通过病原学检查鉴别"],
        "diff2": ["症状位置不同", "淋巴瘤常有全身淋巴结肿大"],
        "diff3": ["已知原发肿瘤病史", "突发胸痛、低氧血症"],
        "method": ["病理学检查", "免疫组化染色", "基因检测"],
        "drug_type": ["抗肿瘤药物", "化疗药物", "靶向药物"],
        "drug2": ["卡铂", "紫杉醇", "吉西他滨"],
        "interaction": ["可能增加骨髓抑制风险", "需注意肝肾毒性叠加"],
        "factor": ["肝肾功能", "体表面积", "基因检测结果"],
        "monitor_items": ["血常规", "肝肾功能", "心电图"],
    }

    texts = []
    for _ in range(num_samples):
        category = random.choice(list(templates.keys()))
        template = random.choice(templates[category])
        text = f"# {category}\n\n{template}"
        while "{" in text:
            start = text.index("{")
            end = text.index("}") + 1
            key = text[start+1:end-1]
            text = text[:start] + random.choice(fillers.get(key, ["?" + key + "?"])) + text[end:]
        texts.append(text)

    random.shuffle(texts)
    split = int(len(texts) * 0.95)
    train_texts, val_texts = texts[:split], texts[split:]

    Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
    (Path(DATA_DIR) / "train.txt").write_text("\n\n===SEP===\n\n".join(train_texts), encoding="utf-8")
    (Path(DATA_DIR) / "val.txt").write_text("\n\n===SEP===\n\n".join(val_texts), encoding="utf-8")

    print(f"样本数据生成完成: 训练集 {len(train_texts)} 篇, 验证集 {len(val_texts)} 篇")
    print(f"train.txt: {(Path(DATA_DIR)/'train.txt').stat().st_size:,} 字节")
    print(f"val.txt:   {(Path(DATA_DIR)/'val.txt').stat().st_size:,} 字节")

generate_sample_data(100)

## 3. Dataset 与 DataLoader

In [ ]:
class MedicalTextDataset(Dataset):
    def __init__(self, data_path: str, tokenizer, max_length: int = 512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        with open(data_path, "r", encoding="utf-8") as f:
            text = f.read()
        segments = text.split("===SEP===")
        self.examples = []
        for seg in segments:
            seg = seg.strip()
            if len(seg) < 10:
                continue
            enc = tokenizer(seg, truncation=True, max_length=max_length, return_tensors=None)
            self.examples.append({"input_ids": enc["input_ids"], "labels": enc["input_ids"].copy()})

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_fn(batch):
    input_ids = torch.nn.utils.rnn.pad_sequence(
        [torch.tensor(item["input_ids"]) for item in batch],
        batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence(
        [torch.tensor(item["labels"]) for item in batch],
        batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "labels": labels,
            "attention_mask": (input_ids != 0).long()}

## 4. 加载模型与 LoRA 配置

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if DEVICE.type == "cuda" else torch.float32,
    device_map="auto" if DEVICE.type == "cuda" else None,
)
model = model.to(DEVICE)

print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"可训练参数: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
tokenizer.model_max_length = MAX_LENGTH
train_dataset = MedicalTextDataset(f"{DATA_DIR}/train.txt", tokenizer, MAX_LENGTH)
val_dataset = MedicalTextDataset(f"{DATA_DIR}/val.txt", tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn)

print(f"训练样本: {len(train_dataset)}, 验证样本: {len(val_dataset)}")
print(f"训练批次/epoch: {len(train_loader)}, 验证批次: {len(val_loader)}")

## 5. 训练循环

训练过程:
- 每 10 步评估验证集 loss
- 自动保存最佳模型
- 每个 epoch 生成样本文本

In [ ]:
def evaluate(model, val_loader, device):
    model.eval()
    total_loss = 0.0
    num_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(input_ids=batch["input_ids"].to(device),
                            labels=batch["labels"].to(device),
                            attention_mask=batch["attention_mask"].to(device))
            total_loss += outputs.loss.item()
            num_batches += 1
    model.train()
    return total_loss / max(num_batches, 1)


def generate_sample(model, tokenizer, prompt, device, max_new_tokens=80):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     do_sample=True, temperature=0.7, top_p=0.9)
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    model.train()
    return generated

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.05 * total_steps),
    num_training_steps=total_steps,
)

global_step = 0
best_val_loss = float("inf")
train_loss_log = []
val_loss_log = []
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        outputs = model(input_ids=input_ids, labels=labels, attention_mask=attention_mask)
        loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % 10 == 0:
                val_loss = evaluate(model, val_loader, DEVICE)
                val_loss_log.append((global_step, val_loss))
                avg_train_loss = epoch_loss / (step + 1)
                train_loss_log.append((global_step, avg_train_loss))

                elapsed = time.time() - start_time
                print(f"Epoch {epoch}/{EPOCHS} | Step {global_step:06d} | "
                      f"Train loss: {avg_train_loss:.4f} | Val loss: {val_loss:.4f} | "
                      f"LR: {scheduler.get_last_lr()[0]:.2e} | Time: {elapsed:.0f}s")

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    model.save_pretrained(f"{OUTPUT_DIR}/best_model")
                    tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")
                    print(f"  -> 保存最佳模型 (val_loss={val_loss:.4f})")

    for prompt in ["临床表现：", "诊断依据：", "治疗方案：", "预后判断："]:
        sample = generate_sample(model, tokenizer, prompt, DEVICE)
        print(f"\n[Epoch {epoch} 生成样本 | prompt: {prompt}]")
        print(sample)

model.save_pretrained(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")

records = {"train_loss": train_loss_log, "val_loss": val_loss_log,
            "best_val_loss": best_val_loss,
            "total_time_seconds": time.time() - start_time,
            "config": {"model": MODEL_NAME, "lora_r": LORA_R, "epochs": EPOCHS}}
json.dump(records, open(f"{OUTPUT_DIR}/training_log.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

print(f"\n训练完成! 总耗时: {(time.time()-start_time)/60:.2f} 分钟")
print(f"最佳验证 loss: {best_val_loss:.4f}")

## 6. 推理: 加载最佳模型生成医学文本

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True,
    torch_dtype=torch.bfloat16 if DEVICE.type == "cuda" else torch.float32,
)
model_infer = PeftModel.from_pretrained(base_model, f"{OUTPUT_DIR}/best_model")
model_infer = model_infer.to(DEVICE)
model_infer.eval()

prompts = ["临床表现：", "诊断依据：", "治疗方案：", "预后判断：",
           "鉴别诊断：", "并发症：", "用药原则：", "出院标准："]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = model_infer.generate(**inputs, max_new_tokens=200,
                                           do_sample=True, temperature=0.7, top_p=0.9)
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    new_content = generated[len(prompt):].strip()
    print(f"【{prompt}】")
    print(f"  {new_content[:300]}")
    print()